
# Citi Bike — Where We Go From Here

This notebook is a **continuation notebook** for `CitiBike_2025_EDA_Modeling_v6.ipynb`.

It does two jobs:

1. **Extract and summarize the results already embedded in v6** so the story is easy to explain.
2. **Turn those results into concrete next experiments** that move the project closer to the original goal: predicting bike/scooter availability, not just next-hour demand.

---

## The current state in one sentence

Your modeling pipeline is already strong enough to claim that **next-hour departures and arrivals are predictable at the busiest stations**, but the project is **not finished** because:

- the current evaluation is on only the **top 50 busiest stations**,
- the **capacity merge appears broken**,
- the utilization/availability proxy is currently **all NaN**, and
- the project goal is still **availability**, while the best current target is **next-hour demand**.


In [1]:

from pathlib import Path
import io
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Markdown, display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

SOURCE_NOTEBOOK = Path("CitiBike_2025_EDA_Modeling_v6.ipynb")
FEATURE_CACHE_CANDIDATES = [
    Path("outputs/feature_df_model_ready_2025_v6.parquet"),
    Path("feature_df_model_ready_2025_v6.parquet"),
    Path("/mnt/data/outputs/feature_df_model_ready_2025_v6.parquet"),
]
STATION_LOOKUP_CANDIDATES = [
    Path("station_lookup.csv"),
    Path("/mnt/data/station_lookup.csv"),
]
GBFS_STATION_INFO_URL = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json"



## 1) Pull the key results out of the v6 notebook

The nice part about notebooks is that their outputs are stored in the `.ipynb` file. This cell reads those stored outputs directly, so we can reuse your existing results even if the cached parquet files are not available yet.


In [2]:
def load_notebook(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Notebook not found: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def cell_source(cell):
    return "".join(cell.get("source", []))


def first_stream_text(cell):
    texts = []
    for out in cell.get("outputs", []):
        if out.get("output_type") == "stream":
            texts.append("".join(out.get("text", [])))
    return "\n".join(texts)


def html_tables_from_cell(cell):
    tables = []
    for out in cell.get("outputs", []):
        if out.get("output_type") in {"display_data", "execute_result"}:
            data = out.get("data", {})
            html = data.get("text/html")
            if html:
                html_str = "".join(html) if isinstance(html, list) else html
                try:
                    parsed = pd.read_html(io.StringIO(html_str))
                    tables.extend(parsed)
                except Exception:
                    pass
    return tables


def find_cell(nb, needle: str):
    for cell in nb["cells"]:
        if needle in cell_source(cell):
            return cell
    raise KeyError(f"Could not find cell containing: {needle}")


nb_json = load_notebook(SOURCE_NOTEBOOK)

config_cell = find_cell(nb_json, "TOP_N_STATIONS_FOR_MODEL =")
agg_cell = find_cell(nb_json, "station_hour_cache = OUTPUT_DIR")
util_cell = find_cell(nb_json, "def add_capacity_utilization")
summary_cell = find_cell(nb_json, "target_summary = pd.DataFrame(summary_rows)")
primary_results_cell = find_cell(nb_json, 'print("Primary target:", PRIMARY_TARGET)')
weather_error_cell = find_cell(nb_json, 'weather_summary = weather_error.groupby("is_precipitating"')
importance_cell = find_cell(nb_json, 'importance_df = extract_feature_importance')
residual_cell = find_cell(nb_json, 'Residual mean:')

config_text = cell_source(config_cell)
agg_text = first_stream_text(agg_cell)
util_text = first_stream_text(util_cell)
residual_text = first_stream_text(residual_cell)

summary_tables = html_tables_from_cell(summary_cell)
primary_tables = html_tables_from_cell(primary_results_cell)
weather_tables = html_tables_from_cell(weather_error_cell)
importance_tables = html_tables_from_cell(importance_cell)

results = {
    "top_n_stations": int(re.search(r"TOP_N_STATIONS_FOR_MODEL\s*=\s*(\d+)", config_text).group(1)),
    "agg_unique_stations": int(re.search(r"Unique stations:\s*(\d+)", agg_text).group(1)),
    "agg_rows": int(re.search(r"\((\d+),\s*6\)", agg_text).group(1)),
    "capacity_matched": int(re.search(r"Capacity matched:\s*(\d+)/(\d+)", agg_text).group(1)),
    "capacity_total": int(re.search(r"Capacity matched:\s*(\d+)/(\d+)", agg_text).group(2)),
    "target_summary": summary_tables[0] if summary_tables else None,
    "primary_validation": primary_tables[0] if len(primary_tables) >= 1 else None,
    "primary_test": primary_tables[1] if len(primary_tables) >= 2 else None,
    "weather_error": weather_tables[0] if weather_tables else None,
    "feature_importance": importance_tables[0] if importance_tables else None,
    "residual_text": residual_text,
    "util_text": util_text,
}

results["target_summary"]



## 2) What the current notebook already proves

These are the strongest claims you can already make from v6:

- **Random Forest (deeper)** is the best model for all three current targets.
- **Departures** and **arrivals** are much easier to predict than **net flow**.
- The strongest signals are mostly **recent activity** and **time-of-day / cyclic time features**.
- Weather appears to matter, but it looks **secondary** compared with lagged demand and time structure.
- The current system is best described as a **next-hour demand model for busy stations**, not yet a true citywide availability model.


In [3]:

print(f"Top-N stations used for modeling: {results['top_n_stations']}")
print(f"Unique stations in aggregated station-hour table: {results['agg_unique_stations']}")
print(f"Station-hour rows in aggregation: {results['agg_rows']:,}")
print(f"Capacity matched during merge: {results['capacity_matched']}/{results['capacity_total']}")
print()

if results["target_summary"] is not None:
    display(Markdown("### Current best-model summary"))
    display(results["target_summary"])

if results["primary_test"] is not None:
    display(Markdown("### Primary target test-set comparison"))
    display(results["primary_test"])

if results["feature_importance"] is not None:
    display(Markdown("### Top feature importances for the primary target"))
    display(results["feature_importance"].head(15))

if results["weather_error"] is not None:
    display(Markdown("### Dry vs precipitating error slice"))
    display(results["weather_error"])

print("Residual diagnostics from v6:")
print(results["residual_text"].strip())
print()
print("Capacity / utilization text from v6:")
print(results["util_text"].strip())


Top-N stations used for modeling: 50
Unique stations in aggregated station-hour table: 2428
Station-hour rows in aggregation: 11,966,828
Capacity matched during merge: 0/2428

Residual diagnostics from v6:
Residual mean: -1.716  |  std: 5.331  |  skew: 0.312

Capacity / utilization text from v6:
Utilization ratio stats:



## 3) The biggest gaps to close next

### Gap A — Coverage
The current model uses only the **top 50 busiest stations**, while the aggregation contains **2,428 stations**. That is a perfectly reasonable modeling choice, but it means the current scores are **not citywide scores yet**.

### Gap B — Capacity / utilization
The capacity merge reports **0 / 2,428 stations matched**, and the utilization summary shows **all NaN**. That means the notebook currently cannot support the availability-proxy part of the story yet.

### Gap C — Task mismatch
Your original project framing is about **bike availability**, but your strongest result so far is on **departures/arrivals next hour**. That is still a good result, but the final project should explicitly bridge from demand prediction to an availability or shortage-risk task.

### Gap D — Evaluation story
The next report will be much stronger if you can show:

1. how accuracy changes as you move from top 50 to top 100 / 250 / all stations,
2. which feature families actually help,
3. where the model fails most often, and
4. how the model supports a real operational decision like **shortage alerts** or **rebalancing**.



## 4) Experiment 1 — Audit and fix the capacity join

This is the highest-leverage cleanup because it unblocks a true availability proxy.

The likely issue is a **station-key mismatch** between the station lookup data and the GBFS feed. This cell standardizes station IDs, checks overlap before and after normalization, and creates a merge-ready capacity table.


In [4]:

import requests


def normalize_station_key(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\s+", "", regex=True)
    return s


def locate_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None


def fetch_gbfs_capacity(url: str = GBFS_STATION_INFO_URL) -> pd.DataFrame:
    payload = requests.get(url, timeout=30).json()
    stations = payload["data"]["stations"]
    gbfs = pd.DataFrame(stations)

    keep_cols = [c for c in ["station_id", "short_name", "name", "capacity", "lat", "lon"] if c in gbfs.columns]
    gbfs = gbfs[keep_cols].copy()
    gbfs["station_id_norm"] = normalize_station_key(gbfs["station_id"])
    if "short_name" in gbfs.columns:
        gbfs["short_name_norm"] = normalize_station_key(gbfs["short_name"])
    else:
        gbfs["short_name_norm"] = np.nan
    return gbfs


station_lookup_path = locate_existing_path(STATION_LOOKUP_CANDIDATES)

if station_lookup_path is None:
    print("station_lookup.csv not found. Add it beside this notebook to run the capacity audit.")
else:
    station_lookup = pd.read_csv(station_lookup_path)
    gbfs_capacity = fetch_gbfs_capacity()

    # Try to detect a likely station ID column.
    possible_id_cols = [
        "station_id", "short_name", "station_key", "id", "ride_id", "station_code"
    ]
    possible_id_cols = [c for c in possible_id_cols if c in station_lookup.columns]
    if not possible_id_cols:
        raise ValueError(f"Could not find a likely station ID column in station_lookup.csv. Columns: {station_lookup.columns.tolist()}")

    station_id_col = possible_id_cols[0]
    station_lookup = station_lookup.copy()
    station_lookup["station_id_norm"] = normalize_station_key(station_lookup[station_id_col])

    raw_overlap = station_lookup[station_id_col].astype(str).isin(gbfs_capacity["station_id"].astype(str)).mean()
    norm_overlap_station_id = station_lookup["station_id_norm"].isin(gbfs_capacity["station_id_norm"]).mean()
    norm_overlap_short_name = station_lookup["station_id_norm"].isin(gbfs_capacity["short_name_norm"].dropna()).mean()

    audit = pd.DataFrame({
        "match_strategy": [
            "raw station_lookup id vs GBFS station_id",
            "normalized station_lookup id vs normalized GBFS station_id",
            "normalized station_lookup id vs normalized GBFS short_name",
        ],
        "match_rate": [raw_overlap, norm_overlap_station_id, norm_overlap_short_name],
    })
    display(audit)

    merged_on_station_id = station_lookup.merge(
        gbfs_capacity[["station_id_norm", "capacity"]].drop_duplicates(),
        on="station_id_norm",
        how="left",
    )

    merged_on_short_name = station_lookup.merge(
        gbfs_capacity[["short_name_norm", "capacity"]].drop_duplicates(),
        left_on="station_id_norm",
        right_on="short_name_norm",
        how="left",
    )

    print("Capacity non-null after normalized station_id merge:", merged_on_station_id["capacity"].notna().sum())
    print("Capacity non-null after normalized short_name merge:", merged_on_short_name["capacity"].notna().sum())


,match_strategy,match_rate
0,raw station_lookup id vs GBFS station_id,0.000000
1,normalized station_lookup id vs normalized GBF...,0.000000
2,normalized station_lookup id vs normalized GBF...,0.940692


Capacity non-null after normalized station_id merge: 0
Capacity non-null after normalized short_name merge: 2284



## 5) Load the cached feature table if it exists

The next experiments are easiest if you already have the cached feature parquet from v6:

`outputs/feature_df_model_ready_2025_v6.parquet`

If it is not found, this notebook still gives you the roadmap and code skeletons, but the modeling cells will skip execution.


In [5]:

feature_df_path = locate_existing_path(FEATURE_CACHE_CANDIDATES)
feature_df = None

if feature_df_path is None:
    print("feature_df cache not found. Re-run v6 locally until it saves outputs/feature_df_model_ready_2025_v6.parquet, then come back here.")
else:
    print("Loading feature_df from:", feature_df_path)
    feature_df = pd.read_parquet(feature_df_path)
    if "station_key" in feature_df.columns:
        feature_df["station_key"] = feature_df["station_key"].astype("category")
    print(feature_df.shape)
    display(feature_df.head())


Loading feature_df from: outputs\feature_df_model_ready_2025_v6.parquet
(11966828, 66)


,station_key,hour,departures,arrivals,net_flow,total_activity,station_name,lat,lng,capacity,is_active_station,location_id,temperature_f,precipitation_in,rain_in,snowfall_in,snow_depth_ft,wind_speed_mph,soil_temperature_0_to_7cm (°F),soil_temperature_7_to_28cm (°F),relative_humidity_pct,dew_point_f,apparent_temperature_f,wind_gusts_mph,wind_direction_deg,hour_of_day,day_of_week,month,is_weekend,is_rush_hour,is_commute_peak,is_overnight,hour_sin,hour_cos,dow_sin,dow_cos,is_holiday,is_precipitating,heavy_precip_flag,is_raining,is_snowing,departures_lag_1,departures_lag_2,departures_lag_24,departures_roll_3,departures_roll_24,arrivals_lag_1,arrivals_lag_2,arrivals_lag_24,arrivals_roll_3,arrivals_roll_24,net_flow_lag_1,net_flow_lag_2,net_flow_lag_24,net_flow_roll_3,net_flow_roll_24,total_activity_lag_1,total_activity_lag_2,total_activity_lag_24,total_activity_roll_3,total_activity_roll_24,departures_next_hour,arrivals_next_hour,net_flow_next_hour,utilization_ratio,utilization_ratio_lag_1
0,1234.56,2025-06-18 22:00:00,0,1,1,1,Morgan HCT Charging,40.708948,-73.932777,NaN,0,1.0,70.2,0.004,0.004,0.0,0.0,0.8,72.5,69.8,100.0,70.1,77.8,5.8,214.0,22,2,6,0,0,0,0,-5.000000e-01,0.866025,0.974928,-0.222521,0,1,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.0,NaN,NaN
1,1234.56,2025-06-18 23:00:00,1,1,0,2,Morgan HCT Charging,40.708948,-73.932777,NaN,0,3.0,71.4,0.004,0.004,0.0,0.0,2.7,72.5,69.4,98.0,71.0,78.8,7.6,195.0,23,2,6,0,0,0,0,-2.588190e-01,0.965926,0.974928,-0.222521,0,1,0,1,0,0.0,NaN,NaN,0.000000,0.000000,1.0,NaN,NaN,1.000000,1.000000,1.0,NaN,NaN,1.000000,1.0,1.0,NaN,NaN,1.000000,1.000000,1.0,0.0,-1.0,NaN,NaN
2,1234.56,2025-06-19 23:00:00,1,0,-1,1,Morgan HCT Charging,40.708948,-73.932777,NaN,0,0.0,73.0,0.000,0.000,0.0,0.0,10.6,74.5,73.6,83.0,67.5,75.2,23.3,234.0,23,3,6,0,0,0,0,-2.588190e-01,0.965926,0.433884,-0.900969,1,0,0,0,0,1.0,0.0,NaN,0.500000,0.500000,1.0,1.0,NaN,1.000000,1.000000,0.0,1.0,NaN,0.500000,0.5,2.0,1.0,NaN,1.500000,1.500000,1.0,1.0,0.0,NaN,NaN
3,1234.56,2025-06-20 09:00:00,1,1,0,2,Morgan HCT Charging,40.708948,-73.932777,NaN,0,3.0,72.6,0.000,0.000,0.0,0.0,11.3,71.7,71.2,55.0,55.5,69.6,26.8,284.0,9,4,6,0,1,1,0,7.071068e-01,-0.707107,-0.433884,-0.900969,0,0,0,0,0,1.0,1.0,NaN,0.666667,0.666667,0.0,1.0,NaN,0.666667,0.666667,-1.0,0.0,NaN,0.000000,0.0,1.0,2.0,NaN,1.333333,1.333333,1.0,1.0,0.0,NaN,NaN
4,1234.56,2025-06-20 12:00:00,1,1,0,2,Morgan HCT Charging,40.708948,-73.932777,NaN,0,2.0,79.3,0.000,0.000,0.0,0.0,11.2,78.9,72.3,46.0,56.8,79.9,27.7,286.0,12,4,6,0,0,0,0,1.224647e-16,-1.000000,-0.433884,-0.900969,0,0,0,0,0,1.0,1.0,NaN,1.000000,0.750000,1.0,0.0,NaN,0.666667,0.750000,0.0,-1.0,NaN,-0.333333,0.0,2.0,1.0,NaN,1.666667,1.500000,1.0,1.0,0.0,NaN,NaN



## 6) Experiment 2 — Coverage vs accuracy tradeoff

This is the cleanest way to answer a very important question:

> Are the good scores only true for the busiest stations, or do they hold up as we expand coverage?

We keep the current task (`departures_next_hour`) but vary the number of stations included.


In [6]:

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "rmse": float(np.sqrt(mse)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


def chronological_split(model_df):
    model_df = model_df.sort_values("hour").reset_index(drop=True)
    unique_hours = np.sort(model_df["hour"].unique())
    train_idx = max(1, int(len(unique_hours) * 0.70))
    valid_idx = max(train_idx + 1, int(len(unique_hours) * 0.85))
    valid_idx = min(valid_idx, len(unique_hours) - 1)

    train_cutoff = unique_hours[train_idx]
    valid_cutoff = unique_hours[valid_idx]

    train_df = model_df.loc[model_df["hour"] < train_cutoff].copy()
    valid_df = model_df.loc[(model_df["hour"] >= train_cutoff) & (model_df["hour"] < valid_cutoff)].copy()
    test_df = model_df.loc[model_df["hour"] >= valid_cutoff].copy()
    return train_df, valid_df, test_df


def busiest_station_keys(df, top_n=50):
    return (
        df.groupby("station_key", observed=True)["total_activity"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )


def build_preprocessor(feature_cols, categorical_cols=("station_key", "station_cluster")):
    categorical_cols = [c for c in categorical_cols if c in feature_cols]
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_cols,
            ),
        ],
        remainder="drop",
    )


def train_rf_regressor(train_df, valid_df, test_df, target_col, feature_cols):
    pre = build_preprocessor(feature_cols)
    model = RandomForestRegressor(
        n_estimators=180,
        max_depth=16,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )
    pipe = Pipeline([("preprocess", pre), ("model", model)])

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    X_valid = valid_df[feature_cols]
    y_valid = valid_df[target_col]
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    pipe.fit(X_train, y_train)
    valid_pred = pipe.predict(X_valid)
    test_pred = pipe.predict(X_test)

    valid_metrics = regression_metrics(y_valid, valid_pred)
    test_metrics = regression_metrics(y_test, test_pred)
    return pipe, valid_pred, test_pred, valid_metrics, test_metrics


In [7]:

PRIMARY_TARGET = "departures_next_hour"

base_feature_cols = [
    "station_key",
    "station_cluster",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "hour_of_day", "day_of_week", "month",
    "is_weekend", "is_holiday", "is_rush_hour", "is_commute_peak", "is_overnight",
    "is_precipitating", "is_raining", "is_snowing", "heavy_precip_flag",
    "departures_lag_1", "departures_lag_2", "departures_lag_24",
    "arrivals_lag_1", "arrivals_lag_2", "arrivals_lag_24",
    "net_flow_lag_1", "net_flow_lag_2", "net_flow_lag_24",
    "total_activity_lag_1", "total_activity_lag_2", "total_activity_lag_24",
    "departures_roll_3", "departures_roll_24",
    "arrivals_roll_3", "arrivals_roll_24",
    "net_flow_roll_3", "net_flow_roll_24",
    "total_activity_roll_3", "total_activity_roll_24",
    "lat", "lng", "capacity", "is_active_station", "utilization_ratio_lag_1",
]
base_feature_cols += [c for c in (feature_df.columns if feature_df is not None else []) if str(c).startswith("weather_pc_")]
base_feature_cols = [c for c in base_feature_cols if feature_df is not None and c in feature_df.columns]

if feature_df is None:
    print("Skipping: feature_df not loaded.")
else:
    sensitivity_rows = []
    for top_n in [25, 50, 100, 250]:
        busy = busiest_station_keys(feature_df, top_n=top_n)
        model_df = feature_df.loc[
            feature_df["station_key"].isin(busy) & feature_df[PRIMARY_TARGET].notna(),
            ["hour", PRIMARY_TARGET] + base_feature_cols,
        ].copy()

        for cat_col in [c for c in ["station_key", "station_cluster"] if c in model_df.columns]:
            model_df[cat_col] = model_df[cat_col].astype("category")

        train_df, valid_df, test_df = chronological_split(model_df)
        _, _, _, valid_metrics, test_metrics = train_rf_regressor(
            train_df, valid_df, test_df, PRIMARY_TARGET, base_feature_cols
        )

        sensitivity_rows.append({
            "top_n_stations": top_n,
            "n_rows": len(model_df),
            "valid_rmse": valid_metrics["rmse"],
            "valid_r2": valid_metrics["r2"],
            "test_rmse": test_metrics["rmse"],
            "test_r2": test_metrics["r2"],
        })

    topn_results = pd.DataFrame(sensitivity_rows)
    display(topn_results)

    plt.figure(figsize=(8, 4))
    plt.plot(topn_results["top_n_stations"], topn_results["test_r2"], marker="o")
    plt.title("Coverage vs accuracy for departures_next_hour")
    plt.xlabel("Number of busiest stations modeled")
    plt.ylabel("Test R²")
    plt.grid(True, alpha=0.3)
    plt.show()


KeyboardInterrupt: 


## 7) Experiment 3 — Feature ablation

This tells you **what is actually doing the work**.

A strong report does not just say “the model performed well.” It says:

- lags alone got us most of the signal,
- calendar/cyclic time added more,
- weather added a smaller but real gain,
- capacity/utilization helped only after the merge was fixed.


In [ ]:

FEATURE_GROUPS = {
    "lags_and_rolls": [
        "departures_lag_1", "departures_lag_2", "departures_lag_24",
        "arrivals_lag_1", "arrivals_lag_2", "arrivals_lag_24",
        "net_flow_lag_1", "net_flow_lag_2", "net_flow_lag_24",
        "total_activity_lag_1", "total_activity_lag_2", "total_activity_lag_24",
        "departures_roll_3", "departures_roll_24",
        "arrivals_roll_3", "arrivals_roll_24",
        "net_flow_roll_3", "net_flow_roll_24",
        "total_activity_roll_3", "total_activity_roll_24",
    ],
    "calendar": [
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "hour_of_day", "day_of_week", "month",
        "is_weekend", "is_holiday", "is_rush_hour", "is_commute_peak", "is_overnight",
    ],
    "weather": [
        "is_precipitating", "is_raining", "is_snowing", "heavy_precip_flag",
    ],
    "spatial_and_station": [
        "station_key", "station_cluster", "lat", "lng",
    ],
    "capacity_related": [
        "capacity", "is_active_station", "utilization_ratio_lag_1",
    ],
}


def available_features(df, cols):
    return [c for c in cols if c in df.columns]


def run_ablation(df, target_col="departures_next_hour", top_n=50):
    busy = busiest_station_keys(df, top_n=top_n)

    experiment_defs = [
        ("lags only", FEATURE_GROUPS["lags_and_rolls"]),
        ("lags + calendar", FEATURE_GROUPS["lags_and_rolls"] + FEATURE_GROUPS["calendar"]),
        ("lags + calendar + weather", FEATURE_GROUPS["lags_and_rolls"] + FEATURE_GROUPS["calendar"] + FEATURE_GROUPS["weather"]),
        ("lags + calendar + weather + spatial", FEATURE_GROUPS["lags_and_rolls"] + FEATURE_GROUPS["calendar"] + FEATURE_GROUPS["weather"] + FEATURE_GROUPS["spatial_and_station"]),
        ("full", FEATURE_GROUPS["lags_and_rolls"] + FEATURE_GROUPS["calendar"] + FEATURE_GROUPS["weather"] + FEATURE_GROUPS["spatial_and_station"] + FEATURE_GROUPS["capacity_related"]),
    ]

    rows = []
    trained = {}
    for label, features in experiment_defs:
        features = available_features(df, list(dict.fromkeys(features + [c for c in df.columns if str(c).startswith("weather_pc_")])))
        model_df = df.loc[
            df["station_key"].isin(busy) & df[target_col].notna(),
            ["hour", target_col] + features,
        ].copy()

        for cat_col in [c for c in ["station_key", "station_cluster"] if c in model_df.columns]:
            model_df[cat_col] = model_df[cat_col].astype("category")

        train_df, valid_df, test_df = chronological_split(model_df)
        pipe, valid_pred, test_pred, valid_metrics, test_metrics = train_rf_regressor(
            train_df, valid_df, test_df, target_col, features
        )

        rows.append({
            "experiment": label,
            "n_features": len(features),
            "valid_rmse": valid_metrics["rmse"],
            "valid_r2": valid_metrics["r2"],
            "test_rmse": test_metrics["rmse"],
            "test_r2": test_metrics["r2"],
        })
        trained[label] = {
            "model": pipe,
            "features": features,
            "train_df": train_df,
            "valid_df": valid_df,
            "test_df": test_df,
            "test_pred": test_pred,
        }

    return pd.DataFrame(rows), trained


if feature_df is None:
    print("Skipping: feature_df not loaded.")
else:
    ablation_results, trained_runs = run_ablation(feature_df, target_col=PRIMARY_TARGET, top_n=50)
    display(ablation_results)

    plt.figure(figsize=(10, 4))
    plt.plot(ablation_results["experiment"], ablation_results["test_r2"], marker="o")
    plt.xticks(rotation=20, ha="right")
    plt.title("Feature ablation for departures_next_hour")
    plt.ylabel("Test R²")
    plt.grid(True, alpha=0.3)
    plt.show()



## 8) Experiment 4 — Error slicing

This section answers *where* the model struggles:

- rush hour vs non-rush hour,
- rainy vs dry,
- low-demand vs spike-demand periods,
- different station clusters.

That gives you much stronger discussion material than reporting one average RMSE.


In [ ]:

if feature_df is None:
    print("Skipping: feature_df not loaded.")
elif 'trained_runs' not in globals():
    print("Run the ablation cell first.")
else:
    best_label = ablation_results.sort_values("test_r2", ascending=False).iloc[0]["experiment"]
    best_run = trained_runs[best_label]
    test_df = best_run["test_df"].copy()
    test_df["prediction"] = best_run["test_pred"]
    test_df["abs_error"] = (test_df[PRIMARY_TARGET] - test_df["prediction"]).abs()

    if "hour" in test_df.columns:
        test_df["hour_of_day"] = pd.to_datetime(test_df["hour"]).dt.hour
    if PRIMARY_TARGET in test_df.columns:
        test_df["demand_bucket"] = pd.qcut(test_df[PRIMARY_TARGET].rank(method="first"), q=5, labels=["very low", "low", "mid", "high", "very high"])

    slice_tables = {}
    for col in ["hour_of_day", "is_precipitating", "is_weekend", "is_commute_peak", "station_cluster", "demand_bucket"]:
        if col in test_df.columns:
            slice_tables[col] = test_df.groupby(col, as_index=False)["abs_error"].mean().sort_values("abs_error", ascending=False)

    for name, table in slice_tables.items():
        display(Markdown(f"### MAE by `{name}`"))
        display(table.head(20))

    if "hour_of_day" in slice_tables:
        plt.figure(figsize=(10, 4))
        plt.plot(slice_tables["hour_of_day"]["hour_of_day"], slice_tables["hour_of_day"]["abs_error"])
        plt.title("MAE by hour of day")
        plt.xlabel("Hour of day")
        plt.ylabel("MAE")
        plt.grid(True, alpha=0.3)
        plt.show()



## 9) Experiment 5 — Reframe the project as an operational alert task

A practical deployment often does **not** need an exact count. It needs an answer to questions like:

- Will this station be **very busy** next hour?
- Is this station at **shortage risk** next hour?
- Should the system **rebalance** bikes here soon?

This cell creates a high-demand alert target using each station's own 90th percentile, then trains a classifier.


In [ ]:

def build_high_demand_label(df, target_col="departures_next_hour", quantile=0.90):
    thresholds = df.groupby("station_key", observed=True)[target_col].quantile(quantile)
    out = df.copy()
    out["high_demand_next_hour"] = out.apply(
        lambda row: int(row[target_col] >= thresholds.loc[row["station_key"]]) if pd.notna(row[target_col]) else np.nan,
        axis=1,
    )
    return out, thresholds


def train_rf_classifier(train_df, valid_df, test_df, target_col, feature_cols):
    pre = build_preprocessor(feature_cols)
    clf = RandomForestClassifier(
        n_estimators=180,
        max_depth=16,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )
    pipe = Pipeline([("preprocess", pre), ("model", clf)])

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    X_valid = valid_df[feature_cols]
    y_valid = valid_df[target_col]
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    pipe.fit(X_train, y_train)
    valid_prob = pipe.predict_proba(X_valid)[:, 1]
    test_prob = pipe.predict_proba(X_test)[:, 1]
    valid_pred = (valid_prob >= 0.5).astype(int)
    test_pred = (test_prob >= 0.5).astype(int)

    metrics = pd.DataFrame([
        {
            "split": "validation",
            "precision": precision_score(y_valid, valid_pred, zero_division=0),
            "recall": recall_score(y_valid, valid_pred, zero_division=0),
            "f1": f1_score(y_valid, valid_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_valid, valid_prob),
        },
        {
            "split": "test",
            "precision": precision_score(y_test, test_pred, zero_division=0),
            "recall": recall_score(y_test, test_pred, zero_division=0),
            "f1": f1_score(y_test, test_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
        },
    ])
    return pipe, metrics


if feature_df is None:
    print("Skipping: feature_df not loaded.")
else:
    clf_df, thresholds = build_high_demand_label(feature_df, target_col=PRIMARY_TARGET, quantile=0.90)
    busy = busiest_station_keys(clf_df, top_n=50)
    clf_features = available_features(clf_df, list(dict.fromkeys(
        FEATURE_GROUPS["lags_and_rolls"] + FEATURE_GROUPS["calendar"] + FEATURE_GROUPS["weather"] + FEATURE_GROUPS["spatial_and_station"]
    )))

    model_df = clf_df.loc[
        clf_df["station_key"].isin(busy) & clf_df["high_demand_next_hour"].notna(),
        ["hour", "high_demand_next_hour"] + clf_features,
    ].copy()

    for cat_col in [c for c in ["station_key", "station_cluster"] if c in model_df.columns]:
        model_df[cat_col] = model_df[cat_col].astype("category")

    train_df, valid_df, test_df = chronological_split(model_df)
    _, clf_metrics = train_rf_classifier(train_df, valid_df, test_df, "high_demand_next_hour", clf_features)
    display(clf_metrics)



## 10) Optional bridge back to true availability

Once the capacity merge works, you can create an **availability proxy** by combining:

- station capacity,
- cumulative net flow,
- and a starting-fill assumption.

This is still a proxy, but it is much closer to your original project question.


In [ ]:

def add_availability_proxy(df, init_fill_ratio=0.50):
    needed = {"station_key", "hour", "net_flow", "capacity"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns for availability proxy: {missing}")

    out = df[["station_key", "hour", "net_flow", "capacity"]].copy()
    out = out.sort_values(["station_key", "hour"]).copy()

    out["capacity"] = pd.to_numeric(out["capacity"], errors="coerce")
    out["initial_bikes"] = out["capacity"] * init_fill_ratio
    out["cumulative_net_flow"] = out.groupby("station_key", observed=True)["net_flow"].cumsum()
    out["bikes_available_proxy"] = (out["initial_bikes"] + out["cumulative_net_flow"]).clip(lower=0)
    out["bikes_available_proxy"] = np.minimum(out["bikes_available_proxy"], out["capacity"])
    out["availability_ratio_proxy"] = out["bikes_available_proxy"] / out["capacity"].replace(0, np.nan)
    return out


if feature_df is None:
    print("Skipping: feature_df not loaded.")
elif "capacity" not in feature_df.columns or feature_df["capacity"].notna().sum() == 0:
    print("Skip for now: capacity is still missing. Fix the capacity join first.")
else:
    availability_proxy = add_availability_proxy(feature_df, init_fill_ratio=0.50)
    display(availability_proxy.head())

    plt.figure(figsize=(8, 4))
    availability_proxy["availability_ratio_proxy"].dropna().hist(bins=40)
    plt.title("Distribution of proxy availability ratio")
    plt.xlabel("availability_ratio_proxy")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    plt.show()



## 11) Recommended project story from here

If I were turning this into the next checkpoint or the final report, I would tell the story in this order:

### What you already accomplished
- You built a station-hour pipeline from Citi Bike trips, weather, and station metadata.
- You showed that **Random Forest** beats linear regression and the naive baseline.
- You showed that **departures** and **arrivals** are predictably better targets than **net flow**.
- You identified that the most important features are **recent activity** and **time-of-day structure**.

### What you learned
- Predicting a difference signal like **net flow** is harder than predicting departures or arrivals directly.
- The current model is strongest on **busy stations**, which is realistic for a first milestone.
- Weather matters, but the biggest predictive lift appears to come from **lags + calendar/time**.

### What must happen next
1. **Fix the capacity merge** so availability-proxy features actually work.
2. **Measure generalization beyond top 50 stations**.
3. **Run feature ablations** to show what helps.
4. **Add an operational target** like high-demand alert or shortage-risk classification.
5. Optionally, **reconstruct proxy availability** once capacity is fixed.

### The strongest final claim you are aiming for
> We can forecast next-hour Citi Bike demand at the busiest stations with good accuracy, identify when/where the model struggles, and extend the pipeline toward an actionable shortage-risk or availability-proxy system for rebalancing decisions.



## 12) My recommended priority order

If you only have time for a few more improvements, do them in this order:

1. **Fix capacity matching**
2. **Run top-N station sensitivity**
3. **Run feature ablation**
4. **Add high-demand classification**
5. **Add proxy availability / shortage risk**

That sequence gives you the biggest report improvement for the least extra work.
